# EntroPy — End-to-End Factor Research Walkthrough

> **Purpose**: Demonstrate the full production-style multi-factor research pipeline.
> This notebook is designed to be shown in interviews or used as a template for new factor research.

## Research Question

> *"Can entropy-based and state-space signals (Kalman velocity, spectral entropy, Hurst exponent) provide verifiable incremental alpha beyond classic cross-sectional factors (momentum, volatility, liquidity) — after realistic transaction costs, multiple-testing controls, and out-of-sample validation?"*

## Pipeline

```
Data  →  Factor Compute  →  Single-Factor Tearsheets  →  Multiple-Testing
     →  Redundancy Clustering  →  Multi-Factor Combiner  →  Decision Story
     →  Production Readiness Score  →  Recommendation
```

---

**Estimated runtime**: 5–15 minutes (synthetic data demo mode).  
**Real data**: Replace `build_synthetic_panel()` with `load_parquet('data/prices/prices.parquet')`.

In [ ]:
import sys, warnings
from pathlib import Path

# Make the project importable from inside the notebooks/ directory
_root = Path().resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from quant_platform.core.utils.io import set_project_root
set_project_root(_root)

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

print('✓ Imports OK — project root:', _root)

## §0  Synthetic Data (Replace with Real Prices)

For demo purposes we generate a synthetic price panel with **500 stocks × 1500 trading days** (~6 years). Each stock follows a GBM with mild cross-sectional variation in drift and volatility.

In [ ]:
def build_synthetic_panel(n_stocks: int = 80, n_days: int = 600, seed: int = 0) -> pd.DataFrame:
    """Synthetic price + market_cap panel for demo purposes."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range('2019-01-02', periods=n_days, freq='B')
    tickers = [f'S{i:03d}' for i in range(n_stocks)]
    rows = []
    for tk in tickers:
        mu = rng.normal(0.0002, 0.0003)          # drift
        sigma = rng.uniform(0.015, 0.035)          # vol
        base = rng.uniform(10, 500)
        log_ret = rng.normal(mu, sigma, n_days)
        closes = base * np.cumprod(1 + log_ret)
        opens  = closes * rng.uniform(0.997, 1.003, n_days)
        vols   = (closes * rng.uniform(1e6, 5e7, n_days)).astype(int)
        mcap   = closes * rng.uniform(1e7, 1e10, n_days)
        for i, dt in enumerate(dates):
            rows.append({
                'date': dt, 'ticker': tk,
                'open': opens[i], 'high': closes[i] * 1.02,
                'low': closes[i] * 0.98, 'close': closes[i],
                'adj_close': closes[i], 'adj_factor': 1.0,
                'volume': vols[i],
                'amount': closes[i] * vols[i],
                'is_tradable': True,
                'market_cap': mcap[i],
            })
    return pd.DataFrame(rows)


prices = build_synthetic_panel(n_stocks=80, n_days=600)
print(f'Panel: {prices["ticker"].nunique()} stocks × {prices["date"].nunique()} dates')
prices.head(3)

## §1  Compute Factors via the Registry

The `FactorRegistry` auto-discovers all 42 registered factors.  
For demo speed we pick a representative subset of 8 factors.

In [ ]:
from quant_platform.core.signals.registry import FactorRegistry
from quant_platform.core.signals.cross_sectional.evaluation import add_forward_returns

# Select 8 representative factors for this demo
DEMO_FACTORS = [
    'MOM_12_1M',       # Classic 12-1 momentum
    'STR_1M',          # Short-term reversal
    'ILLIQ_AMIHUD',    # Amihud illiquidity
    'VOL_20D',         # 20-day realised volatility
    'GROSS_PROFITABILITY',  # Novy-Marx profitability
    'OVERNIGHT_RET_21D',   # ★ New: overnight return
    'KF_VELOCITY',     # Kalman filtered trend velocity
    'HURST_60D',       # Hurst exponent (trend vs mean-reversion)
]

reg = FactorRegistry()
reg.discover()

print(f'Total registered factors: {len(reg)}')
print('Computing subset:', DEMO_FACTORS)

factor_df = reg.compute_all(prices, factor_names=DEMO_FACTORS)
factor_df = add_forward_returns(prices, periods=[1, 5]).merge(
    factor_df, on=['date', 'ticker'], how='right'
)

print(f'\nFactor panel: {len(factor_df):,} rows, {factor_df.shape[1]} columns')
factor_df[['date', 'ticker'] + DEMO_FACTORS].tail(3)

## §2  Single-Factor Tearsheets (IC, Half-life, Long/Short Attribution)

For each factor we run a comprehensive evaluation including:
- IC and RankIC with **Newey-West HAC** t-stat (corrects for autocorrelation)
- IC decay → **alpha half-life** (drives rebalance frequency decision)
- **Long vs short leg Sharpe** decomposition
- **Cross-sectional stability** across size / liquidity buckets

In [ ]:
from quant_platform.core.signals.cross_sectional.evaluation import (
    factor_tearsheet, compare_factors, estimate_alpha_half_life, ic_decay,
)

tearsheets = {}
for fname in DEMO_FACTORS:
    if fname not in factor_df.columns:
        print(f'  skip {fname} (not computed)')
        continue
    direction = reg.get(fname).meta.direction
    ts = factor_tearsheet(
        factor_df, fname,
        direction=direction,
        forward_periods=[1, 5],
        cost_bps_per_turnover=10.0,
    )
    tearsheets[fname] = ts
    adv = ts['advanced_metrics']
    ric = ts['rank_ic_stats']
    print(f'{fname:25s}  RankIC={ric["mean_ic"]:+.4f}  '
          f'NW-t={ric["nw_t_stat"]:+.2f}  '
          f'HL={adv.get("alpha_half_life_days", float("nan")):.1f}d  '
          f'LO={str(adv.get("long_only_compatible", "?"))[0]}')

print(f'\n✓ Computed tearsheets for {len(tearsheets)} factors')

In [ ]:
# Build comparison table
comparison = compare_factors(tearsheets)
comparison[['ric_mean_ic', 'ric_icir', 'ric_nw_t_stat', 'cost_adj_ls_sharpe',
            'break_even_cost_bps', 'mean_turnover', 'alpha_half_life_days',
            'long_leg_sharpe', 'short_leg_sharpe', 'long_only_compatible'
           ]].round(4)

### 2a  NW-adjusted IC significance

Compare i.i.d. vs Newey-West HAC t-statistics.  
**Key insight**: factors evaluated on 5d/10d forward returns have overlapping observations → the i.i.d. t-stat overstates significance.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
factors_list = [f for f in DEMO_FACTORS if f in tearsheets]
iid_t  = [tearsheets[f]['rank_ic_stats'].get('t_stat', np.nan)    for f in factors_list]
nw_t   = [tearsheets[f]['rank_ic_stats'].get('nw_t_stat', np.nan) for f in factors_list]
x = np.arange(len(factors_list))
w = 0.35

ax1.bar(x - w/2, iid_t, w, label='i.i.d. t-stat', color='#3498db', alpha=0.8)
ax1.bar(x + w/2, nw_t,  w, label='NW HAC t-stat', color='#e74c3c', alpha=0.8)
ax1.axhline(2.0,  color='black', linestyle='--', linewidth=1, label='t=2.0 threshold')
ax1.axhline(-2.0, color='black', linestyle='--', linewidth=1)
ax1.set_xticks(x); ax1.set_xticklabels(factors_list, rotation=30, ha='right', fontsize=9)
ax1.set_ylabel('t-statistic'); ax1.set_title('i.i.d. vs Newey-West HAC IC t-stat')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3, axis='y')

half_lives = [tearsheets[f]['advanced_metrics'].get('alpha_half_life_days', np.nan) for f in factors_list]
valid = [(f, hl) for f, hl in zip(factors_list, half_lives) if np.isfinite(hl) and hl < 50]
if valid:
    fnames_hl, hl_vals = zip(*valid)
    bars = ax2.bar(range(len(fnames_hl)), hl_vals, color='#9b59b6', alpha=0.8)
    ax2.set_xticks(range(len(fnames_hl)))
    ax2.set_xticklabels(fnames_hl, rotation=30, ha='right', fontsize=9)
    ax2.axhline(5,  color='green', linestyle='--', linewidth=1, label='5d')
    ax2.axhline(21, color='orange',linestyle='--', linewidth=1, label='21d (monthly rebal)')
    ax2.set_ylabel('Half-life (days)'); ax2.set_title('Alpha Decay Half-life → Rebalance Guidance')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## §3  Multiple-Testing Controls

Running 8 (or 42) factors on the same dataset inflates false discoveries.  
We apply **Benjamini-Hochberg FDR** (q ≤ 0.10) and the **Hansen SPA test**.

In [ ]:
from quant_platform.core.signals.factor_selection import (
    apply_multiple_testing_controls, apply_deployability_filters, hansen_spa_test,
)

# Apply all multiple-testing controls
comparison_mt = apply_multiple_testing_controls(comparison, tearsheets)
comparison_mt = apply_deployability_filters(comparison_mt)

# Hansen SPA test (single joint test for the best factor)
ls_rets = {f: tearsheets[f]['long_short'] for f in tearsheets if 'long_short' in tearsheets[f]}
spa = hansen_spa_test(ls_rets, n_boot=1000)
print(f"Hansen SPA (consistent):  p = {spa['spa_pvalue_c']:.3f}  →  "
      f"{'Reject H₀: best factor beats zero' if spa['spa_pvalue_c'] < 0.10 else 'Fail to reject H₀'}")
print(f"Best strategy by SPA: {spa['best_strategy']}")

keep_cols = ['ric_mean_ic', 'nw_t_stat', 'fdr_q_value', 'fdr_pass_10pct',
             'deflated_ls_sharpe', 'deployable', 'deployability_score']
display_cols = [c for c in keep_cols if c in comparison_mt.columns]
comparison_mt[display_cols].round(4)

## §4  Redundancy Pruning → Hierarchical Clustering

Even significant factors may carry overlapping information.  
We use **hierarchical agglomerative clustering** on `1 - |corr|` distance and pick the best-scoring representative per cluster.

In [ ]:
from quant_platform.core.signals.redundancy import (
    build_redundancy_report, select_complementary_factors,
    cluster_based_factor_selection, RedundancyConfig,
)

direction_map = {f: reg.get(f).meta.direction for f in DEMO_FACTORS if f in reg}

red_report = build_redundancy_report(
    factor_df, DEMO_FACTORS,
    direction_map=direction_map,
    return_col='fwd_ret_1d',
)

# ── Greedy selection (legacy) ──
greedy_selected = select_complementary_factors(
    comparison_mt if 'deployability_score' in comparison_mt.columns else comparison,
    red_report,
    config=RedundancyConfig(max_factors=5),
)
greedy_factors = greedy_selected[greedy_selected['selected']]['factor'].tolist()

# ── Hierarchical clustering selection (new) ──
cluster_selected = cluster_based_factor_selection(
    comparison_mt if not comparison_mt.empty else comparison,
    red_report,
    n_clusters=4,
    score_metric='deployability_score',
)
cluster_factors = cluster_selected[cluster_selected['selected']]['factor'].tolist()

print('Greedy selection:     ', greedy_factors)
print('Clustering selection: ', cluster_factors)

cluster_selected[['factor', 'cluster_id', 'cluster_size', 'selection_score', 'selected', 'cluster_members']]

In [ ]:
# ── Visualise signal correlation matrix ──
sig_corr = red_report.get('signal_correlation', pd.DataFrame())
if not sig_corr.empty:
    fig, ax = plt.subplots(figsize=(8, 6))
    n = len(sig_corr)
    im = ax.imshow(sig_corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(sig_corr.columns, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(sig_corr.index, fontsize=9)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{sig_corr.values[i,j]:.2f}', ha='center', va='center',
                    fontsize=8, color='black' if abs(sig_corr.values[i,j]) < 0.7 else 'white')
    plt.colorbar(im, ax=ax)
    ax.set_title('Effective Signal Correlation (Spearman)', fontsize=11, fontweight='bold')
    plt.tight_layout(); plt.show()

## §5  Production Readiness Score (PRS)

Each factor receives a 0-100 score across 7 dimensions.

In [ ]:
from quant_platform.core.evaluation.production_readiness import (
    score_factor_catalog, score_factor_readiness, format_prs_report,
)

prs_df = score_factor_catalog(tearsheets, comparison_mt)
print(prs_df[['factor','total','verdict','significance','stability',
              'economic','capacity','crowding','width','implementation']].to_string(index=False))

In [ ]:
# Detailed PRS report for the top-ranked factor
best_factor = prs_df.iloc[0]['factor']
prs_detail = score_factor_readiness(
    tearsheets[best_factor]['advanced_metrics'],
    tearsheets[best_factor]['ic_stats'],
    tearsheets[best_factor]['rank_ic_stats'],
    factor_name=best_factor,
)
print(format_prs_report(prs_detail))

In [ ]:
# ── PRS visualisation ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of PRS totals
ax = axes[0]
verdict_colors = {'ACCEPT': '#2ecc71', 'CONDITIONAL': '#f39c12',
                  'WATCHLIST': '#95a5a6', 'REJECT': '#e74c3c'}
colors = [verdict_colors.get(v, '#aaa') for v in prs_df['verdict']]
ax.barh(prs_df['factor'], prs_df['total'], color=colors, alpha=0.85)
ax.axvline(80, color='#2ecc71', linestyle='--', linewidth=1.2, label='ACCEPT ≥ 80')
ax.axvline(60, color='#f39c12', linestyle='--', linewidth=1.2, label='CONDITIONAL ≥ 60')
ax.set_xlabel('Production Readiness Score (0-100)')
ax.set_title('PRS Ranking', fontweight='bold')
ax.invert_yaxis(); ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='x')

# Radar for best factor
import math as _math
ax2 = axes[1]
from quant_platform.core.evaluation.production_readiness import DIMENSION_WEIGHTS
dims = list(DIMENSION_WEIGHTS.keys())
scores = [prs_detail['breakdown'][d]['score'] for d in dims]
labels = [d.replace('_', '\n').title() for d in dims]
n = len(dims)
angles = [i / n * 2 * _math.pi for i in range(n)] + [0]
scores_p = scores + [scores[0]]
ax2 = plt.subplot(122, polar=True)
ax2.plot(angles, scores_p, 'o-', color='#1a3a5c', linewidth=2)
ax2.fill(angles, scores_p, alpha=0.25, color='#1a3a5c')
ax2.set_xticks(angles[:-1]); ax2.set_xticklabels(labels, size=9)
ax2.set_ylim(0, 100)
ax2.set_title(f'PRS Radar: {best_factor}', size=10, fontweight='bold', pad=20)

plt.tight_layout(); plt.show()

## §6  Multi-Factor Combiner Comparison

Compare 4 data-driven combination methods on the selected factors.

In [ ]:
from quant_platform.core.alpha_models.multi_factor import (
    MultiFactorCombiner, MultiFactorConfig, infer_factor_metadata,
)

# Use clustering-selected factors (or all if clustering returned too few)
final_factors = cluster_factors if len(cluster_factors) >= 2 else DEMO_FACTORS[:4]
directions, categories = infer_factor_metadata(final_factors)

combiner_oos_sharpes = {}

for method in ['rolling_icir', 'mean_variance', 'risk_parity', 'orthogonal_incremental']:
    try:
        cfg = MultiFactorConfig(
            method=method,
            lookback=126,
            baseline_factors=tuple(final_factors[:2]),   # first 2 as baseline for orthogonal
        )
        combiner = MultiFactorCombiner(cfg, direction_map=directions, category_map=categories)
        combined = combiner.fit_transform(factor_df, final_factors, return_col='fwd_ret_1d')

        # OOS: last 20% of dates
        all_dates = sorted(combined['date'].unique())
        oos_start = all_dates[int(len(all_dates) * 0.8)]
        oos = combined[combined['date'] >= oos_start].copy()
        oos_merged = oos.merge(factor_df[['date','ticker','fwd_ret_1d']], on=['date','ticker'], how='left')

        from quant_platform.core.signals.cross_sectional.evaluation import long_short_returns
        ls = long_short_returns(oos_merged, 'alpha_multi', return_col='fwd_ret_1d')
        oos_sharpe = float(ls.mean() / ls.std() * np.sqrt(252)) if len(ls) > 10 else np.nan
        combiner_oos_sharpes[method] = oos_sharpe
        print(f'{method:30s}  OOS Sharpe = {oos_sharpe:+.3f}')
    except Exception as e:
        print(f'{method:30s}  ERROR: {e}')
        combiner_oos_sharpes[method] = np.nan

In [ ]:
# Plot combiner comparison
valid = {k: v for k, v in combiner_oos_sharpes.items() if np.isfinite(v)}
if valid:
    fig, ax = plt.subplots(figsize=(8, 4))
    methods = [k.replace('_', '\n') for k in valid]
    sharpes = list(valid.values())
    colors = ['#2ecc71' if s > 0.5 else '#e67e22' if s > 0 else '#e74c3c' for s in sharpes]
    bars = ax.bar(methods, sharpes, color=colors, alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(0.5, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Sharpe ≥ 0.5')
    for bar, v in zip(bars, sharpes):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.03, f'{v:.2f}',
                ha='center', fontsize=11, fontweight='bold')
    ax.set_ylabel('OOS Annualised Sharpe'); ax.legend(fontsize=9)
    ax.set_title('Multi-Factor Combiner OOS Comparison', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## §7  Generate HTML Tearsheets & Decision Story

In [ ]:
from quant_platform.core.evaluation.tearsheet import generate_factor_tearsheet, generate_all_tearsheets
from quant_platform.core.evaluation.report_story import generate_decision_story

OUT_DIR = Path('..') / 'data' / 'reports'

# ── Single tearsheet for best factor ──
ts_html = generate_factor_tearsheet(
    tearsheets[best_factor],
    factor_name=best_factor,
    output_path=OUT_DIR / f'tearsheet_{best_factor}.html',
    extra_meta={'Data': 'Synthetic demo', 'Stocks': 80, 'Days': 600},
)
print(f'Tearsheet saved → {OUT_DIR / f"tearsheet_{best_factor}.html"}')

# ── Decision story for all factors ──
story_html = generate_decision_story(
    tearsheets,
    comparison=comparison_mt,
    selected_factors=cluster_factors,
    combiner_results=combiner_oos_sharpes,
    redundancy_report=red_report,
    output_path=OUT_DIR / 'decision_story.html',
    experiment_name='Demo Factor Research',
)
print(f'Decision story saved → {OUT_DIR / "decision_story.html"}')

## §8  Walk-forward OOS Validation

Estimate out-of-sample Sharpe across rolling train/test windows to confirm the signal does not overfit.

In [ ]:
from quant_platform.core.evaluation.walkforward import run_walk_forward, WalkForwardConfig

wf_config = WalkForwardConfig(train_months=18, test_months=6, step_months=6, min_train_obs=80)
wf_results = run_walk_forward(factor_df, prices, best_factor, config=wf_config)

if not wf_results.empty:
    print(wf_results[['fold','train_end','test_start','test_end','train_ic','train_icir',
                       'oos_sharpe','oos_return','oos_max_dd']].to_string(index=False))
    mean_oos = wf_results['oos_sharpe'].mean()
    std_oos  = wf_results['oos_sharpe'].std()
    print(f'\nWF OOS Sharpe: {mean_oos:.3f} ± {std_oos:.3f}')

## §9  Backtest Overfitting Check (CSCV)

Combinatorially Symmetric Cross-Validation (Bailey et al. 2017).  
PBO > 0.5 = selection is essentially noise.  PBO < 0.1 = strong evidence of real edge.

In [ ]:
from quant_platform.core.evaluation.overfit import probability_of_backtest_overfitting

# Build return panel: one column per factor
ls_panel = pd.concat(
    {f: tearsheets[f]['long_short'] for f in tearsheets},
    axis=1,
).dropna(how='all').fillna(0.0)

cscv = probability_of_backtest_overfitting(ls_panel, n_splits=8)
print(f"CSCV PBO = {cscv['pbo']:.3f}")
print(f"  Interpretation: {'Low overfitting risk' if cscv['pbo'] < 0.3 else 'Moderate overfitting' if cscv['pbo'] < 0.5 else 'High overfitting risk — selection likely noise'}")
print(f"  n_combinations evaluated: {cscv['n_combinations']}")
print(f"  Stochastic dominance: {cscv['stochastic_dominance']:.3f}")

# Plot logit distribution
if 'logits' in cscv and cscv['logits']:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.hist(cscv['logits'], bins=20, color='#3498db', alpha=0.75, edgecolor='white')
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--', label='PBO threshold (logit=0)')
    pbo_share = sum(l <= 0 for l in cscv['logits']) / len(cscv['logits'])
    ax.set_xlabel('OOS Logit (negative = IS-best underperforms OOS median)')
    ax.set_ylabel('Count'); ax.legend(fontsize=9)
    ax.set_title(f'CSCV Logit Distribution   PBO = {pbo_share:.3f}', fontweight='bold')
    plt.tight_layout(); plt.show()

## §10  Summary & Research Narrative

Fill in your own numbers below after running with real data.

In [ ]:
n_total  = len(tearsheets)
n_pass   = int((prs_df['verdict'].isin(['ACCEPT','CONDITIONAL'])).sum())
best_prs = prs_df.iloc[0]

print('=' * 60)
print('  FACTOR RESEARCH SUMMARY')
print('=' * 60)
print(f'  Factors evaluated:      {n_total}')
print(f'  Pass FDR / gates:       {n_pass}')
print(f'  Selected (clustering):  {len(cluster_factors)}')
print(f'  Best factor (PRS):      {best_prs["factor"]} — PRS {best_prs["total"]:.0f} [{best_prs["verdict"]}]')
mean_oos_sh = wf_results['oos_sharpe'].mean() if not wf_results.empty else float('nan')
print(f'  Best WF OOS Sharpe:     {mean_oos_sh:.3f}')
print(f'  CSCV PBO:               {cscv["pbo"]:.3f}')
print('=' * 60)
print()
print('  Selected factor set:', cluster_factors)
print()
print('  Recommended combiner: orthogonal_incremental')
print('  Rationale: Tests whether advanced signals add true incremental')
print('  alpha on top of classic CS baseline.')
print('=' * 60)

In [ ]:
print('\n📄 Open in browser:')
print(f'   {(OUT_DIR / f"tearsheet_{best_factor}.html").resolve()}')
print(f'   {(OUT_DIR / "decision_story.html").resolve()}')